# 2 — Convert Lawrence Police PDFs to pipeline-compatible CSV (legacy + 2026)

This notebook retains the original 2023–2025 extraction functions and adds a dedicated parser for the redesigned 2026 press logs. **Run the configuration cell last** after inspecting the examples and validation.

**Contract:** `Incident #, Date, Type, Location, Arrested, Name, DOB, Charges, Disposition`. The first eight columns match the original converter; `Disposition` is appended last.


## Decisions and rationale

1. **Keep the old parser:** legacy `Incident # / Date / Type / Location / Arrested / Name / DOB / Charges` continues to use the original regex extraction; a small postprocessing fix removes a stray `Location` token accidentally captured by its `Type` regex.
2. **Detect layout from the PDF text:** 2026 pages contain `CFS:` and `Disposition:`. This avoids making assumptions from filenames or dates.
3. **Use `pdfplumber` first, OCR only as fallback:** the provided 2026 file already contains readable text; OCR would introduce avoidable recognition errors.
4. **2026 CFS → Type:** use the descriptive part after ` - `, uppercased. This preserves the *call-for-service category* in the old Type column; it does **not** claim to identify a charge.
5. **2026 Disposition → Arrested:** `ARREST` maps to `Yes`, all six other known dispositions map to `No` (meaning no arrest reported by this disposition, **not** proof no arrest ever occurred). Preserve the original disposition in the last column.
6. **Do not invent missing information:** the redesigned report does not give names, DOBs or charges even for `ARREST`; leave these blank. Legacy rows get a blank Disposition because the field was absent.
7. **Normalize date format:** 2026 `06/03/2026 08:38` becomes `2026-06-03 08:38:00`. The seconds are **not measured**; they are added solely for compatibility.
8. **Keep Location unchanged at conversion:** downstream cleaning and geocoding are responsible for location normalization. Do not strip business names or street prefixes here.
9. **Fail on unparseable PDFs:** a conversion failure should not silently produce an incomplete CSV.




## Original imports and legacy extraction logic (preserved)


In [8]:
#GOES THROUGH ALL DATES FOR ALL YEARS, NOT JUST 2023-2024
import pandas as pd
import os
import pdfplumber
import pytesseract
from pdf2image import convert_from_path
import csv
import re
from html import unescape

# regex patterns
incident_pattern = r"Incident #:\s*(\d+)"
date_pattern = r"Date:\s*(\d{4}-\d{2}-\d{2}\s*\d{2}:\d{2}:\d{2})"
type_pattern = r"Type:\s*([\w\s/]+)"
location_pattern = r"Location:\s*(.+?)(?=\n|$)"
arrest_pattern = r"Arrested:"
name_pattern = r"Name:\s*([^:\n]+?)(?=\s*Date of Birth:)"
dob_pattern = r"Date of Birth:\s*(\d{2}/\d{2}/\d{4})"
charges_pattern = r"Charges:\s*((?:.+?(\n|$))*?)(?=\n(?:\w+:|$))"

# helper: check if the text seems valid
def is_meaningful_police_log(text):
    incident_count = len(re.findall(r"Incident\s+#", text))
    has_keywords = "Location:" in text or "Type:" in text or "NOISE ORD" in text
    is_not_all_symbols = bool(re.search(r"[A-Za-z]{3,}", text))
    return incident_count > 0 and has_keywords and is_not_all_symbols

# extract from pdfplumber format
def extract_data_from_text_pdfplumber(text):
    rows = []
    if not re.search(r"[=-]{10,}", text):
        return rows
    incidents = re.split(r"[=-]{10,}", text)
    for entry in incidents:
        rows.append(extract_entry(entry))
    return rows

# extract from OCR fallback format
def extract_data_from_text_ocr(text):
    rows = []
    incidents = re.split(r"(?=Incident\s+#?:\s*\d+)", text)
    for entry in incidents:
        rows.append(extract_entry(entry))
    return rows

# generic extractor from an entry block
def extract_entry(entry):
    return {
        "Incident #": re.search(incident_pattern, entry).group(1) if re.search(incident_pattern, entry) else "",
        "Date": re.search(date_pattern, entry).group(1) if re.search(date_pattern, entry) else "",
        "Type": re.search(type_pattern, entry).group(1) if re.search(type_pattern, entry) else "",
        "Location": re.search(location_pattern, entry).group(1) if re.search(location_pattern, entry) else "",
        "Arrested": "Yes" if re.search(arrest_pattern, entry) else "No",
        "Name": re.search(name_pattern, entry).group(1).strip() if re.search(name_pattern, entry) else "",
        "DOB": re.search(dob_pattern, entry).group(1) if re.search(dob_pattern, entry) else "",
        "Charges": "; ".join(
            line.strip() for line in re.search(charges_pattern, entry, re.DOTALL).group(1).splitlines()
            if line.strip()
        ) if re.search(charges_pattern, entry, re.DOTALL) else ""
    }



## 2025-6 layout parser and shared extraction orchestration

The new format sometimes wraps CFS values over multiple lines and sometimes prints column headers on a separate line. Each incident ends with its `Location:` line, which provides a stable record boundary in the sample.


In [9]:

# New 2025-6 press-log layout. The existing legacy extractor above is unchanged.
# Output column order intentionally matches the old CSV, with Disposition appended.
OUTPUT_COLUMNS = ["Incident #", "Date", "Type", "Location", "Arrested", "Name", "DOB", "Charges", "Disposition"]
DISPOSITIONS = (
    "CLOSED NO DISPATCH", "SERVICED - SEE NOTES", "Merged to Primary",
    "NO CAR AVAILABLE", "REPORT REQUIRED", "ARREST", "CITED"
)
DISPOSITION_RE = re.compile("|".join(re.escape(x) for x in sorted(DISPOSITIONS, key=len, reverse=True)), re.I)
DATE_RE_2026 = re.compile(r"\b(\d{2}/\d{2}/\d{4})\s+(\d{2}:\d{2})\b")
INCIDENT_RE_2026 = re.compile(r"\b\d{8}\b")


def is_new_press_log(text):
    return "CFS:" in text and "Disposition:" in text


def normalize_cfs(cfs):
    """Keep the call category, not a fabricated criminal charge.
    Prefer the descriptive portion after ' - ' (e.g. 'Noise Ord - Noise Ordinance').
    """
    cfs = re.sub(r"\s+", " ", cfs).strip()
    if " - " in cfs:
        cfs = cfs.split(" - ", 1)[1].strip()
    return cfs.upper()


def parse_new_page(page_text):
    """Read 2026 entries as blocks ending at their Location line.
    Works with inline and vertically wrapped table columns in pdfplumber text.
    """
    rows = []
    lines = [line.strip() for line in page_text.splitlines() if line.strip()]
    block = []
    for line in lines:
        if line.startswith("User:") or line.startswith("Powered by") or re.match(r"^Page \d+ of \d+", line):
            continue
        if line.startswith("Location:"):
            block.append(line)
            row = parse_new_block(block)
            if row:
                rows.append(row)
            block = []
        else:
            block.append(line)
    return rows


def parse_new_block(lines):
    location_lines = [line.split("Location:", 1)[1].strip() for line in lines if line.startswith("Location:")]
    if not location_lines:
        return None
    # Headers can be on a separate line; use the data-bearing text between date and location.
    content = " ".join(line for line in lines if not line.startswith("Location:"))
    date_match = DATE_RE_2026.search(content)
    if not date_match:
        return None
    after_date = content[date_match.end():]
    incident_match = INCIDENT_RE_2026.search(after_date)
    if not incident_match:
        return None
    incident = incident_match.group()
    remainder = after_date[incident_match.end():].strip()
    # Remove optional repeated column labels before extracting values.
    remainder = re.sub(r"^(?:CFS:\s*)", "", remainder, flags=re.I)
    remainder = re.sub(r"\bDisposition:\s*", "", remainder, flags=re.I)
    disposition_match = None
    for match in DISPOSITION_RE.finditer(remainder):
        disposition_match = match
    if disposition_match is None:
        return None
    disposition = disposition_match.group().upper()
    cfs = remainder[:disposition_match.start()].strip()
    cfs = re.sub(r"^CFS:\s*", "", cfs, flags=re.I)
    # pdfplumber may emit header labels ahead of the row in wrapped entries.
    cfs = re.sub(r"^(?:Date:\s*Incident #:\s*CFS:\s*Disposition:\s*)", "", cfs, flags=re.I)
    if not cfs:
        return None
    from datetime import datetime
    date = datetime.strptime(date_match.group(1) + " " + date_match.group(2), "%m/%d/%Y %H:%M")
    return {
        "Incident #": incident,
        "Date": date.strftime("%Y-%m-%d %H:%M:%S"),
        "Type": normalize_cfs(cfs),
        "Location": " ".join(location_lines),
        "Arrested": "Yes" if disposition == "ARREST" else "No",
        "Name": "", "DOB": "", "Charges": "",
        "Disposition": disposition,
    }


def parse_new_text(text):
    # Page boundaries are preserved in extract_data_from_pdf_auto, but the parser
    # also works on concatenated text because each incident ends with Location.
    return parse_new_page(text)


def parse_legacy_text(text, ocr=False):
    rows = extract_data_from_text_ocr(text) if ocr else extract_data_from_text_pdfplumber(text)
    # The old separator parser can create empty rows: discard them.
    rows = [r for r in rows if r.get("Incident #") and r.get("Date")]
    for r in rows:
        # Original type_pattern uses \s and may accidentally capture the next "Location" label.
        # Repair that legacy regex artifact without changing the original extractor.
        r["Type"] = re.sub(r"\s*Location\s*$", "", r["Type"]).strip()
        r["Disposition"] = ""  # The old format did not report this field.
    return rows


def parse_text_by_layout(text, ocr=False):
    if is_new_press_log(text):
        return parse_new_text(text), "2026"
    return parse_legacy_text(text, ocr=ocr), "legacy"


def valid_rows(rows):
    return bool(rows) and all(
        r.get("Incident #") and r.get("Date") and r.get("Type") and r.get("Location")
        for r in rows
    )


def extract_data_from_pdf_auto(pdf_path):
    """Try existing text-based extraction first, then OCR only on failure."""
    try:
        with pdfplumber.open(pdf_path) as pdf:
            text = unescape("\n".join(p.extract_text() or "" for p in pdf.pages))
        rows, layout = parse_text_by_layout(text)
        # Fail closed rather than exporting partial rows from an unrecognized format.
        if valid_rows(rows):
            print(f"   Text extraction ({layout}): {len(rows)} rows")
            return rows
        print(f"⚠️ Text extraction failed validation: {os.path.basename(pdf_path)}; trying OCR")
    except Exception as exc:
        print(f"⚠️ pdfplumber error: {os.path.basename(pdf_path)}: {exc}; trying OCR")

    try:
        images = convert_from_path(pdf_path, dpi=300)
        ocr_text = "\n".join(pytesseract.image_to_string(image) for image in images)
        rows, layout = parse_text_by_layout(unescape(ocr_text), ocr=True)
        if valid_rows(rows):
            print(f"   OCR extraction ({layout}): {len(rows)} rows")
            return rows
    except Exception as exc:
        print(f"❌ OCR failed: {os.path.basename(pdf_path)}: {exc}")
    print(f"❌ No validated records: {os.path.basename(pdf_path)}")
    return []


def parse_all_pdfs_to_csv(input_dir, output_csv, max_pdfs=None):
    if not os.path.isdir(input_dir):
        raise FileNotFoundError(f"Input PDF directory not found: {input_dir}")
    all_rows, pdf_count, failures = [], 0, []
    for root, _, files in os.walk(input_dir):
        for file in sorted(files):
            if not file.lower().endswith(".pdf"):
                continue
            path = os.path.join(root, file)
            print(f"📄 Parsing: {file}")
            rows = extract_data_from_pdf_auto(path)
            if not rows:
                failures.append(path)
            all_rows.extend(rows)
            pdf_count += 1
            if max_pdfs and pdf_count >= max_pdfs:
                break
        if max_pdfs and pdf_count >= max_pdfs:
            break
    if failures:
        print("❌ Conversion failed for:", *failures, sep="\n  ")
    os.makedirs(os.path.dirname(os.path.abspath(output_csv)), exist_ok=True)
    with open(output_csv, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=OUTPUT_COLUMNS)
        writer.writeheader()
        writer.writerows(all_rows)
    print(f"✅ Saved {len(all_rows)} rows from {pdf_count} PDFs to {output_csv}")
    return pd.DataFrame(all_rows, columns=OUTPUT_COLUMNS)


## Test both supplied PDFs before bulk processing

Set paths to the two reference files; these assertions catch obvious schema regressions, missing dispositions, and false charge inference.


In [10]:
LEGACY_TEST_PDF = "/Users/sreej/Desktop/Gateway_lawrence/gatewayinitiative-lawrencepd/scripts/data/2026_march/03-04-2026.pdf"  
NEW_TEST_PDF = "/Users/sreej/Desktop/Gateway_lawrence/gatewayinitiative-lawrencepd/0_RecompiledScripts/data/2026_june/06-03-2026.pdf"    

if os.path.exists(LEGACY_TEST_PDF) and os.path.exists(NEW_TEST_PDF):
    old_rows = extract_data_from_pdf_auto(LEGACY_TEST_PDF)
    new_rows = extract_data_from_pdf_auto(NEW_TEST_PDF)
    assert old_rows and new_rows
    assert all(list(row.keys()) == OUTPUT_COLUMNS for row in new_rows)
    assert all(row["Disposition"] in [x.upper() for x in DISPOSITIONS] for row in new_rows)
    assert all(not row["Name"] and not row["DOB"] and not row["Charges"] for row in new_rows)
    assert any(row["Arrested"] == "Yes" for row in new_rows)
    assert any(row["Arrested"] == "Yes" and row["Charges"] for row in old_rows)
    print("Legacy rows:", len(old_rows), "New rows:", len(new_rows))
    display(pd.DataFrame(new_rows)[OUTPUT_COLUMNS].head(8))
else:
    print("Update test PDF paths to run the two-file smoke test.")


   Text extraction (legacy): 248 rows
   Text extraction (2026): 175 rows
Legacy rows: 248 New rows: 175


,Incident #,Date,Type,Location,Arrested,Name,DOB,Charges,Disposition
0,26025682,2026-06-03 23:29:00,TOW REPO,550 BROADWAY,No,,,,SERVICED - SEE NOTES
1,26025681,2026-06-03 23:17:00,ALARM BURGLARY,"KFC, 490 S UNION ST",No,,,,SERVICED - SEE NOTES
2,26025679,2026-06-03 23:07:00,NOISE ORDINANCE,97 PARK ST,No,,,,SERVICED - SEE NOTES
3,26025678,2026-06-03 22:59:00,SUSPICIOUS PERSON,19 SPRINGFIELD ST,No,,,,SERVICED - SEE NOTES
4,26025676,2026-06-03 22:19:00,NOISE ORDINANCE,111 COOLIDGE ST,No,,,,SERVICED - SEE NOTES
5,26025675,2026-06-03 22:17:00,EXTRA CHECK,62 BEACON ST,No,,,,SERVICED - SEE NOTES
6,26025674,2026-06-03 22:13:00,TOW TRESPASS,2 INMAN ST,No,,,,SERVICED - SEE NOTES
7,26025673,2026-06-03 22:08:00,ASSAULT & BATTERY,"WALGREENS, 135 BROADWAY",No,,,,REPORT REQUIRED


## Run on a chosen folder

Use a **year-specific folder or root** containing only the PDFs intended for this CSV. The original downloader stores PDFs under year/month folders; this function searches recursively. If you point it at all years, the output will combine all years. **Do not run a full dataset until the sample validation passes.**


In [11]:
# Update to your Mac repo location before running:
PDF_DATA_DIR = "/Users/sreej/Desktop/Gateway_lawrence/gatewayinitiative-lawrencepd/0_RecompiledScripts/data"
parsed_csv_path = os.path.join(PDF_DATA_DIR, "lawrence_2026.csv")

# Uncomment when ready to run the complete conversion:
result_df = parse_all_pdfs_to_csv(PDF_DATA_DIR, parsed_csv_path)
display(result_df.head())


📄 Parsing: 04-10-2023.pdf
⚠️ Text extraction failed validation: 04-10-2023.pdf; trying OCR
❌ No validated records: 04-10-2023.pdf
📄 Parsing: 04-10-2023_2C_20AIRPORT_20COMMISSION_20AGENDA.pdf
⚠️ Text extraction failed validation: 04-10-2023_2C_20AIRPORT_20COMMISSION_20AGENDA.pdf; trying OCR
❌ No validated records: 04-10-2023_2C_20AIRPORT_20COMMISSION_20AGENDA.pdf
📄 Parsing: 06-01-2026.pdf
   Text extraction (2026): 199 rows
📄 Parsing: 06-02-2026.pdf
   Text extraction (2026): 138 rows
📄 Parsing: 06-03-2026.pdf
   Text extraction (2026): 175 rows
📄 Parsing: 06-04-2026.pdf
   Text extraction (2026): 155 rows
📄 Parsing: 06-05-2026.pdf
   Text extraction (2026): 246 rows
📄 Parsing: 06-06-2026.pdf
   Text extraction (2026): 97 rows
📄 Parsing: 06-07-2026.pdf
   Text extraction (2026): 231 rows
📄 Parsing: 06-08-2026.pdf
   Text extraction (2026): 170 rows
📄 Parsing: 06-09-2026.pdf
   Text extraction (2026): 96 rows
📄 Parsing: 06-10-2026.pdf
   Text extraction (2026): 164 rows
📄 Parsing: 06-11-

,Incident #,Date,Type,Location,Arrested,Name,DOB,Charges,Disposition
0,26025341,2026-06-01 23:53:00,NOISE ORDINANCE,ALLSTON ST & HOWARD ST,No,,,,SERVICED - SEE NOTES
1,26025340,2026-06-01 23:52:00,SEX OFFENSE,75 MANCHESTER ST,No,,,,REPORT REQUIRED
2,26025338,2026-06-01 23:29:00,EXTRA CHECK,"SOUTH LAWRENCE EAST SCHOOL, 165 CRAWFORD ST",No,,,,SERVICED - SEE NOTES
3,26025337,2026-06-01 23:16:00,911 HANG-UP,98 OSGOOD ST,No,,,,SERVICED - SEE NOTES
4,26025336,2026-06-01 23:15:00,MV STOP,20 ISLAND ST,No,,,,SERVICED - SEE NOTES
